# S2 J5/J6 — Pipeline wolof de bout en bout

Assemble la chaîne complète en `lang="wo"` : `ffmpeg → transcribe (ASR) → retrieve (RAG) → generate (LLM) → frontend → synthesize (TTS)`.

**Principe : importer et instrumenter, ne pas réimplémenter.** Chaque brique est testée dans sa propre cellule, un seul modèle en mémoire à la fois — indispensable en local (8 Go RAM ne tiennent pas les trois modèles ensemble), confortable sur Colab.

**NLU désactivé** (`nlu.enabled: false`) : intent non utilisé, modèle Rasa entraîné en FR. Pas de Rasa à lancer.

**Objectif J5 (fonctionnalité, en local ou Colab)** — la chaîne tourne, dans l'ordre de diagnostic :
1. ASR — Whosper transcrit-il du wolof, ou francise-t-il ?
2. RAG — la bonne fiche remonte-t-elle ? (encodage `ë ñ ó à ŋ` préservé ?)
3. LLM — la réponse reste-t-elle en wolof, sans basculer en français ?
4. Frontend — que transforme-t-il (nombres, `.lower()`) ?
5. TTS — l'audio Kiriku est-il intelligible ?

**Objectif J6 (latence, sur Colab GPU uniquement)** — voir la dernière section. Les latences mesurées en CPU local ne sont pas représentatives.

## 0. Setup — détection Colab / local

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    PROJECT_ROOT = Path("/content/noo-far-pipeline")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/noofar-ia/noo-far-pipeline.git /content/noo-far-pipeline
    else:
        !cd {PROJECT_ROOT} && git pull
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)   # pour que les chemins relatifs (data/, config/) résolvent
print(f"Projet : {PROJECT_ROOT} | Colab : {ON_COLAB}")

In [ ]:
# Dépendances (Colab surtout ; en local elles sont déjà dans l'env noofar)
if ON_COLAB:
    !pip install chromadb rank-bm25 coqui-tts torchcodec num2words --quiet
    # Shim : isin_mps_friendly retiré en transformers v5, importé par coqui-tts (chemin xTTS)
    import torch, transformers.pytorch_utils as pu
    if not hasattr(pu, "isin_mps_friendly"):
        pu.isin_mps_friendly = lambda e, t: torch.isin(e, t)

In [ ]:
# Token HF — Kiriku est un dépôt gated (conditions à accepter sur sa page HF)
from huggingface_hub import login
if ON_COLAB:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
# en local : déjà loggé via le cache HF, rien à faire

## 1. Config — bascule wolof

Vérifie que `config.yaml` est bien en `lang: wo`. Sur Colab, force-le au besoin (le dépôt cloné peut être resté en `fr`).

In [ ]:
import yaml
cfg_path = PROJECT_ROOT / "config" / "config.yaml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
print("lang        :", cfg["lang"])
print("asr.wo      :", cfg["models"]["asr"]["wo"])
print("tts.wo      :", cfg["models"]["tts"]["wo"])
print("llm.wo      :", cfg["models"]["llm"]["wo"])
print("retrieval.wo:", cfg["rag"]["retrieval"]["wo"])
print("nlu.enabled :", cfg.get("nlu", {}).get("enabled"))
assert cfg["lang"] == "wo", "config.yaml n'est pas en lang: wo"

## 2. Audio d'entrée

Un vocal wolof enregistré (ex. via Telegram, exporté en `.ogg`), placé sous `data/test_wo/`.
Choisir une question **couverte par une fiche wolof**, sinon on ne distingue pas « retrieval raté » de « pas de fiche ».

In [ ]:
import subprocess
AUDIO_IN = PROJECT_ROOT / "data" / "test_wo" / "q01.ogg"   # ajuster
assert AUDIO_IN.exists(), f"absent : {AUDIO_IN}"

# Conversion 16 kHz mono (ce que fait pipeline.convert_to_wav16k)
WAV16 = AUDIO_IN.with_suffix(".16k.wav")
subprocess.run(["ffmpeg","-y","-i",str(AUDIO_IN),"-ar","16000","-ac","1",str(WAV16)],
               check=True, capture_output=True)
print("prêt :", WAV16)

## 3. ASR — Whosper

**La question de fond du sprint.** Whosper (`CAYTU/whosper-large`, sort en minuscules) transcrit-il correctement le wolof, ou bascule-t-il en français / charabia ? Comparer à ce que tu as réellement dit.

In [ ]:
from asr.transcribe import transcribe
texte = transcribe(str(WAV16), lang="wo")
print("ASR :", texte)
question = texte   # la transcription alimente la suite

> Si la sortie francise ou déraille : noter dans le journal des ruptures (maillon ASR). Piste : `pipeline` force peut-être `language="french"` en interne — un `generate_kwargs={"language": "..."}` dans `transcribe.py` peut être nécessaire.

**Libérer la mémoire avant de charger le maillon suivant** (crucial en local 8 Go).

In [ ]:
import gc, torch
from asr.transcribe import _models_cache as _asr_cache
_asr_cache.clear(); gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("ASR déchargé")

## 4. Retrieval — hybride, fiches wolof

La bonne fiche remonte-t-elle à partir de la transcription ? Point de rupture silencieux typique : un `ë`/`ñ` perdu entre l'ASR et ChromaDB fait rater la fiche sans erreur.

In [ ]:
from rag.retriever import retrieve
passages = retrieve(question, lang="wo", mode="hybrid")
for doc, meta in passages:
    print(meta.get("source", "?"), "-", doc[:120])

## 5. Génération — LLM

In [ ]:
from rag.generator import generate
reponse = generate(question, passages, lang="wo")
print("REP :", reponse)

> Vérifier que la réponse **reste en wolof de bout en bout** (le LLM généraliste a tendance à basculer en français). Si oui, le prompt wolof tient ; sinon, le durcir.

**Libérer le LLM avant le TTS.**

In [ ]:
from rag.generator import unload_llm
unload_llm()
print("LLM déchargé")

## 6. Frontend + TTS — Kiriku

`preparer_pour_tts` verbalise les nombres, applique le lexique, et **force les minuscules pour Kiriku** (son vocab n'a pas de majuscules → noms propres mutilés en silence sinon). Comparer `brut` et `→ TTS` montre ce que le frontend change.

In [ ]:
from tts.frontend import preparer_pour_tts
from tts.synthesize import synthesize
from IPython.display import Audio, display

texte_tts = preparer_pour_tts(reponse, lang="wo", modele=cfg["models"]["tts"]["wo"])
print("brut  :", reponse)
print("→ TTS :", texte_tts)

audio, sr = synthesize(texte_tts, lang="wo")
print("sr :", sr, "| durée :", round(len(audio)/sr, 2), "s")
display(Audio(audio, rate=sr))

---
## 7. J6 — Chaîne complète et latence (Colab GPU uniquement)

Les cellules ci-dessus testent brique par brique (une en mémoire à la fois). Ci-dessous, la chaîne entière via `process()` — **nécessite d'avoir toute la RAM/VRAM pour les trois modèles**, donc Colab, pas les 8 Go locaux.

Les latences par étape (`asr`, `rag_llm`, `frontend`, `tts`) ne sont exploitables **que sur GPU**. En CPU elles sont réelles mais non représentatives. Consigner le matériel avec les chiffres.

In [ ]:
if not ON_COLAB:
    print("⚠ chaîne complète non testée en local (8 Go RAM). Passer sur Colab.")
else:
    from app.pipeline import process
    r = process(str(AUDIO_IN), lang="wo")
    print("ASR :", r["texte_transcrit"])
    print("REP :", r["reponse_texte"])
    print("TTS :", r["texte_tts"])
    print("OUT :", r["audio_out_path"])
    print("LAT :", r["latences"])
    from IPython.display import Audio, display
    import soundfile as sf
    a, s = sf.read(r["audio_out_path"])
    display(Audio(a, rate=s))

## 8. Journal des ruptures

| Question | Maillon | Ce qui a cassé | Friction ou modèle ? |
|---|---|---|---|
| q01 | | | |

**Friction** = corrigible dans le code (encodage, découpage, prompt, frontend). **Modèle** = demande fine-tuning ou changement de modèle. Ne pas confondre : ça décide où investir.